# 02. Dataset prep pipeline (documented walkthrough)

This notebook **is the documented, inspectable data pipeline** for building the
RAG evaluation sets (the same process as `datasets/build_rag_eval_subset.py`).

## Why this exists
We are **not** fitting a classical ML train/test classifier. We build:

1. **Final eval set** (`rag_eval_subset.csv`) — 450 posts, 150 per class  
2. **Dev slice** (`rag_dev_slice.csv`) — 30 posts, 10 per class, carved from (1)

for suicide / depression risk RAG experiments (`suicidal`, `depression`, `normal`).
**Anxiety is dropped** (out of thesis scope).

## Pipeline stages

```
HF or local CSVs → merge/normalize → drop anxiety → clean/dedupe
    → stratified sample 150/class → rag_eval_subset.csv
    → stratified sample 10/class from final → rag_dev_slice.csv
```

CLI for automation: `python datasets/build_rag_eval_subset.py`  


In [1]:
# Path setup — works from repo root or notebooks/
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / "components" / "config.py").exists():
    root = _cwd
elif (_cwd.parent / "components" / "config.py").exists():
    root = _cwd.parent
else:
    raise RuntimeError(
        "Could not locate project root. Open this notebook from the repo or notebooks/ folder."
    )

sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "retriever"))

from components.config import (
    PROJECT_ROOT,
    PDF_PATH,
    CHUNKS_PATH,
    CHROMA_PATH,
    RAG_EVAL_SUBSET_PATH,
    RAG_DEV_SLICE_PATH,
    RAG_EVAL_LABELS,
    RETRIEVAL_SECTIONS,
    MOOD_DISORDER_PREFIXES,
    DATASET_PATH,
)

print("Project root:", PROJECT_ROOT)
print("PDF:         ", PDF_PATH, "| exists=", PDF_PATH.exists())
print("Chunks:      ", CHUNKS_PATH, "| exists=", CHUNKS_PATH.exists())
print("ChromaDB:    ", CHROMA_PATH, "| exists=", CHROMA_PATH.exists())
print("Final eval:  ", RAG_EVAL_SUBSET_PATH, "| exists=", RAG_EVAL_SUBSET_PATH.exists())
print("Dev slice:   ", RAG_DEV_SLICE_PATH, "| exists=", RAG_DEV_SLICE_PATH.exists())
print("Labels:      ", list(RAG_EVAL_LABELS))
print("Sections:    ", RETRIEVAL_SECTIONS)


Project root: /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026
PDF:          /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/icd_11.pdf | exists= True
Chunks:       /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/icd11_chunks.json | exists= True
ChromaDB:     /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/chroma_db | exists= True
Final eval:   /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/datasets/rag_eval_subset.csv | exists= True
Dev slice:    /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/datasets/rag_dev_slice.csv | exists= True
Labels:       ['suicidal', 'depression', 'normal']
Sections:     ['Essential Features', 'Boundary with Normality']


## Config knobs


In [2]:
from components.config import (
    HF_DATASET_REPO,
    HF_TRAIN_FILE,
    HF_TEST_FILE,
    DATASET_TRAIN_PATH,
    DATASET_TEST_PATH,
    RAG_EVAL_SUBSET_PATH,
    RAG_EVAL_META_PATH,
    RAG_DEV_SLICE_PATH,
    RAG_DEV_META_PATH,
    RAG_EVAL_LABELS,
    RAG_EVAL_EXCLUDE,
    RAG_EVAL_PER_CLASS,
    RAG_EVAL_SEED,
    RAG_DEV_PER_CLASS,
    RAG_DEV_SEED,
    RAG_EVAL_MIN_CHARS,
    RAG_EVAL_MAX_CHARS,
)

print("HF repo:", HF_DATASET_REPO)
print("Train file:", HF_TRAIN_FILE, "| local exists=", DATASET_TRAIN_PATH.exists())
print("Test file: ", HF_TEST_FILE, "| local exists=", DATASET_TEST_PATH.exists())
print("Keep:", RAG_EVAL_LABELS, "| Drop:", RAG_EVAL_EXCLUDE)
print(f"Final: {RAG_EVAL_PER_CLASS}/class seed={RAG_EVAL_SEED} | exists=", RAG_EVAL_SUBSET_PATH.exists())
print(f"Dev:   {RAG_DEV_PER_CLASS}/class seed={RAG_DEV_SEED} | exists=", RAG_DEV_SLICE_PATH.exists())
print(f"Char bounds: [{RAG_EVAL_MIN_CHARS}, {RAG_EVAL_MAX_CHARS}]")


HF repo: ourafla/Mental-Health_Text-Classification_Dataset
Train file: mental_heath_unbanlanced.csv | local exists= False
Test file:  mental_health_combined_test.csv | local exists= False
Keep: ('suicidal', 'depression', 'normal') | Drop: ('anxiety',)
Final: 150/class seed=42 | exists= True
Dev:   10/class seed=43 | exists= True
Char bounds: [40, 2000]


## Stage helpers

The following cell defines the same functions used by
`datasets/build_rag_eval_subset.py`. Keeping them here makes the thesis pipeline
readable without jumping files; the `.py` CLI remains the automation entrypoint.


In [3]:
import hashlib
import json
import re
from pathlib import Path

import pandas as pd

_WHITESPACE_RE = re.compile(r"\s+")
_URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)


def _file_info(path: Path) -> dict:
    if not path.exists():
        return {"path": str(path), "exists": False}
    return {"path": str(path), "exists": True, "size_bytes": path.stat().st_size}


def _normalize_frame(df: pd.DataFrame, source_split: str) -> pd.DataFrame:
    if "status" not in df.columns or "text" not in df.columns:
        raise ValueError(
            f"Expected columns 'text' and 'status' in {source_split}; got {list(df.columns)}"
        )
    return pd.DataFrame(
        {
            "text": df["text"].astype(str),
            "label": df["status"].astype(str).str.strip().str.lower(),
            "source_split": source_split,
        }
    )


def load_raw_frames() -> tuple[pd.DataFrame, dict]:
    """Prefer local CSVs; else download from Hugging Face and cache locally."""
    local_ok = DATASET_TRAIN_PATH.exists() and DATASET_TEST_PATH.exists()
    stats = {
        "load_mode": "local" if local_ok else "huggingface",
        "train_input": _file_info(DATASET_TRAIN_PATH),
        "test_input": _file_info(DATASET_TEST_PATH),
        "hf_repo": HF_DATASET_REPO,
    }
    if local_ok:
        print(f"Loading local CSVs:\n  {DATASET_TRAIN_PATH}\n  {DATASET_TEST_PATH}")
        train_df = pd.read_csv(DATASET_TRAIN_PATH)
        test_df = pd.read_csv(DATASET_TEST_PATH)
    else:
        print(f"Local CSVs missing; downloading {HF_DATASET_REPO} ...")
        from datasets import load_dataset

        ds = load_dataset(
            HF_DATASET_REPO,
            data_files={"train": HF_TRAIN_FILE, "test": HF_TEST_FILE},
        )
        train_df = ds["train"].to_pandas()
        test_df = ds["test"].to_pandas()
        DATASET_TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
        train_df.to_csv(DATASET_TRAIN_PATH, index=False)
        test_df.to_csv(DATASET_TEST_PATH, index=False)
        stats["train_input"] = _file_info(DATASET_TRAIN_PATH)
        stats["test_input"] = _file_info(DATASET_TEST_PATH)

    merged = pd.concat(
        [_normalize_frame(train_df, "train"), _normalize_frame(test_df, "test")],
        ignore_index=True,
    )
    stats["rows_raw"] = int(len(merged))
    stats["label_counts_raw"] = merged["label"].value_counts().to_dict()
    return merged, stats


def clean_text(text: str) -> str:
    text = text.strip()
    text = _URL_RE.sub(" ", text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    return text


def filter_and_clean(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Drop anxiety / invalid labels, clean text, length-filter, dedupe."""
    stats: dict = {}
    exclude = {x.lower() for x in RAG_EVAL_EXCLUDE}
    keep = {x.lower() for x in RAG_EVAL_LABELS}

    before = len(df)
    df = df[~df["label"].isin(exclude)].copy()
    stats["rows_dropped_exclude"] = before - len(df)

    df = df[df["label"].isin(keep)].copy()
    stats["rows_after_label_filter"] = len(df)

    df["text"] = df["text"].map(clean_text)
    before = len(df)
    df = df[df["text"].str.len() > 0].copy()
    stats["rows_dropped_empty"] = before - len(df)

    before = len(df)
    df = df[
        (df["text"].str.len() >= RAG_EVAL_MIN_CHARS)
        & (df["text"].str.len() <= RAG_EVAL_MAX_CHARS)
    ].copy()
    stats["rows_dropped_length"] = before - len(df)
    stats["rows_after_length_filter"] = len(df)

    before = len(df)
    df["_key"] = df["text"].str.lower()
    df = df.drop_duplicates(subset=["_key"], keep="first").drop(columns=["_key"])
    stats["rows_dropped_dedupe"] = before - len(df)
    stats["rows_after_dedupe"] = len(df)
    stats["label_counts_clean"] = df["label"].value_counts().to_dict()
    return df.reset_index(drop=True), stats


def stratified_sample(df, *, per_class: int, seed: int, id_prefix: str) -> pd.DataFrame:
    """Deterministic per-class sample; dev slice keeps parent_row_id from final set."""
    parts = []
    for label in RAG_EVAL_LABELS:
        pool = df[df["label"] == label]
        if len(pool) < per_class:
            raise RuntimeError(
                f"Need {per_class} rows for '{label}', have {len(pool)}"
            )
        sort_cols = [c for c in ("source_split", "text", "row_id") if c in pool.columns]
        pool = pool.sort_values(sort_cols, kind="mergesort")
        parts.append(pool.sample(n=per_class, random_state=seed))

    out = pd.concat(parts, ignore_index=True)
    order = {label: i for i, label in enumerate(RAG_EVAL_LABELS)}
    out["_o"] = out["label"].map(order)
    out = out.sort_values(["_o", "text"], kind="mergesort").drop(columns=["_o"]).reset_index(drop=True)

    if "row_id" in out.columns and id_prefix == "dev":
        out = out.rename(columns={"row_id": "parent_row_id"})
        out.insert(0, "row_id", [f"dev_{i:04d}" for i in range(len(out))])
        return out[["row_id", "parent_row_id", "text", "label", "source_split"]]

    out = out.drop(columns=["row_id"], errors="ignore")
    out.insert(0, "row_id", [f"{id_prefix}_{i:04d}" for i in range(len(out))])
    return out[["row_id", "text", "label", "source_split"]]


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def write_csv_and_meta(frame, csv_path, meta_path, meta: dict) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(csv_path, index=False, lineterminator="\n")
    meta["output_csv"] = str(csv_path)
    meta["output_rows"] = int(len(frame))
    meta["output_label_counts"] = frame["label"].value_counts().to_dict()
    meta["csv_sha256"] = sha256_file(csv_path)
    meta_path.write_text(json.dumps(meta, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(f"Wrote {csv_path} ({len(frame)} rows)")
    print(f"SHA256: {meta['csv_sha256']}")


print("Pipeline helpers ready.")


Pipeline helpers ready.


## Stage 1:  Load + normalise

Merge the HF “train” and “test” CSVs into one pool. We ignore the classical ML
split meaning: both files are just source text for a RAG eval set.

`status` → `label` (lowercased). Tag `source_split` for provenance.


In [4]:
raw, load_stats = load_raw_frames()
print("load_mode:", load_stats["load_mode"])
print("rows_raw:", load_stats["rows_raw"])
print("label counts (raw):")
print(pd.Series(load_stats["label_counts_raw"]).sort_values(ascending=False))
raw.head(3)


Local CSVs missing; downloading ourafla/Mental-Health_Text-Classification_Dataset ...


/Users/omarhalasa/.pyenv/versions/3.12.0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 49612 examples [00:00, 230417.07 examples/s]
Generating test split: 992 examples [00:00, 95596.67 examples/s]


load_mode: huggingface
rows_raw: 50604
label counts (raw):
normal        18639
depression    14754
suicidal      11460
anxiety        5751
dtype: int64


,text,label,source_split
0,oh my gosh,anxiety,train
1,"trouble sleeping, confused mind, restless hear...",anxiety,train
2,"All wrong, back off dear, forward doubt. Stay ...",anxiety,train


## Stage 2:  Filter + clean

1. Drop `anxiety` (and any label outside the keep-list).  
2. Strip / collapse whitespace; remove URLs.  
3. Enforce character bounds.  
4. Case-insensitive exact-text dedupe.

Inspect counts after this stage before sampling.


In [5]:
cleaned, clean_stats = filter_and_clean(raw)
print("Dropped anxiety/other:", clean_stats["rows_dropped_exclude"])
print("After label filter:", clean_stats["rows_after_label_filter"])
print("Dropped empty:", clean_stats["rows_dropped_empty"])
print("Dropped length:", clean_stats["rows_dropped_length"])
print("Dropped dedupe:", clean_stats["rows_dropped_dedupe"])
print("After dedupe:", clean_stats["rows_after_dedupe"])
print("\nClean label counts:")
print(pd.Series(clean_stats["label_counts_clean"]))
cleaned.head(3)


Dropped anxiety/other: 5751
After label filter: 44853
Dropped empty: 1
Dropped length: 7124
Dropped dedupe: 724
After dedupe: 37004

Clean label counts:
depression    14178
normal        12073
suicidal      10753
dtype: int64


,text,label,source_split
0,"Gr gr dreaming of ex crush to be my game, God",normal,train
1,Leaves are also standby in front of the PC ......,normal,train
2,Thank God even though it's just a ride through,normal,train


## Stage 3: Stratified final eval set (450)

Sample **150 per class** with seed `42`. Sort before sampling so re-runs are
byte-identical. This file is what `eval_mode="final"` uses in `multi_class_rag.ipynb`.


In [6]:
RUN_WRITE = False  # True = overwrite committed CSVs; False = inspect in-memory only

subset = stratified_sample(
    cleaned,
    per_class=RAG_EVAL_PER_CLASS,
    seed=RAG_EVAL_SEED,
    id_prefix="rag",
)
print(subset["label"].value_counts().to_string())
subset.head(3)


label
suicidal      150
depression    150
normal        150


,row_id,text,label,source_split
0,rag_0000,(I live in Canada) I was thinking of ending it...,suicidal,train
1,rag_0001,(context: I want to attempt and fail)Not on he...,suicidal,train
2,rag_0002,1: everyone treats me like shit2: everyone tre...,suicidal,train


## Stage 4: Dev slice (30) from the final set

Sample **10 per class** with seed `43` **from the final 450**, not from the raw
pool. That keeps tuning posts a transparent subset of the reporting set
(`parent_row_id`).


In [7]:
dev = stratified_sample(
    subset,
    per_class=RAG_DEV_PER_CLASS,
    seed=RAG_DEV_SEED,
    id_prefix="dev",
)
print(dev["label"].value_counts().to_string())
print("Parent IDs ⊆ final?", set(dev["parent_row_id"]).issubset(set(subset["row_id"])))
dev.head(5)


label
suicidal      10
depression    10
normal        10
Parent IDs ⊆ final? True


,row_id,parent_row_id,text,label,source_split
0,dev_0000,rag_0000,(I live in Canada) I was thinking of ending it...,suicidal,train
1,dev_0001,rag_0030,I am alone and broken I just feel nothing I ca...,suicidal,train
2,dev_0002,rag_0039,"I am pretty much at a low point, maybe not my ...",suicidal,train
3,dev_0003,rag_0057,I hate myself. My head is so messed up and my ...,suicidal,train
4,dev_0004,rag_0067,"I just want out of this world, I have to many ...",suicidal,train


## Stage 5: Write CSV + provenance meta

Meta JSON stores seeds, filters, stage counts, and SHA256 so teammates can
verify they have the same artifact without re-downloading the raw corpus.


In [8]:
eval_meta = {
    "artifact": "rag_eval_subset",
    "seed": RAG_EVAL_SEED,
    "per_class": RAG_EVAL_PER_CLASS,
    "labels": list(RAG_EVAL_LABELS),
    "exclude_labels": list(RAG_EVAL_EXCLUDE),
    "min_chars": RAG_EVAL_MIN_CHARS,
    "max_chars": RAG_EVAL_MAX_CHARS,
    **load_stats,
    **clean_stats,
}
dev_meta = {
    "artifact": "rag_dev_slice",
    "seed": RAG_DEV_SEED,
    "per_class": RAG_DEV_PER_CLASS,
    "labels": list(RAG_EVAL_LABELS),
    "parent_csv": str(RAG_EVAL_SUBSET_PATH),
    "parent_row_ids": dev["parent_row_id"].tolist(),
    "purpose": "prompt/k tuning only; use eval_mode=final for reporting",
}

if RUN_WRITE:
    write_csv_and_meta(subset, RAG_EVAL_SUBSET_PATH, RAG_EVAL_META_PATH, eval_meta)
    dev_meta["parent_sha256"] = eval_meta["csv_sha256"]
    write_csv_and_meta(dev, RAG_DEV_SLICE_PATH, RAG_DEV_META_PATH, dev_meta)
else:
    print("RUN_WRITE=False — in-memory only; committed CSVs unchanged.")


RUN_WRITE=False — in-memory only; committed CSVs unchanged.


## How this connects to experiments

In `multi_class_rag.ipynb`:

```python
CFG["eval_mode"] = "dev"    # 30-post tuning slice
CFG["eval_mode"] = "final"  # 450-post reporting set
```

Workflow: tune prompts / top-k / alpha on **dev**, then lock settings and report
on **final** (no further prompt editing after the switch).
